# BillyBot Legacy Notebook

This is an output-cleared preservation copy of the old BillyBot notebook.

The original BillyBot git history spans July 25, 2017 (`7a88e7d`) through August 5, 2017 (`c6e2d5c`).

Attribution: BillyBot was heavily adapted from Liza Daly's Brobot tutorial/sample code (see `README.md` and `BROBOT-MIT-LICENSE.txt`).

In [ ]:
# The NLTK library is required:
#!conda install nltk
# We will also use TextBlob:
#!pip install textblob
# Furthermore, in order to use TextBlob(bunchOfText).sentences, I
#   needed to issue the following command:
#!python -m textblob.download_corpora  
#   -- if you need this, you will get an error message telling you so
import random
import logging
import os

os.environ['NLTK_DATA'] = os.getcwd() + '/nltk_data'

from textblob import TextBlob
# Original notebook imported FILTER_WORDS from config.py.
# That copied upstream filter list is omitted from this preserved copy.
FILTER_WORDS = set()

class UnacceptableUtteranceException(Exception):
    """Raised when a generated response should be rejected."""
    pass

logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

# Preprocessing
The user is going to input a sentence. It might be messy, like "it mIGht     be MESSY."
We want to first preprocess the text so that BillyBot can understand it best!

Fortunately, NLTK and TextBlob can do the heavy-lifting for us!

In [ ]:
def process_input_text(input_text):
    """
    Removes extraneous white space and returns TextBlob object
    """
    return TextBlob(' '.join(input_text.split()))

In [ ]:
# Example processing step
processed_text = process_input_text('This is   a  sentence. I made a great         example!')
print(processed_text.sentences)

# Grammatical Components of the Input Text
After the input text has been processed, we can begin to understand its content.  
One way to do this is to parse the text into some of its atomic elements: 
* is there a pronoun?
* is there a verb?
* what nouns are present?
* etc


In [ ]:
def find_candidate_parts_of_speech(parsed):
    """
    Given a parsed input (TextBlob object), find the best pronoun, direct noun, adjective, 
    and verb to match their input.
    Returns a tuple of pronoun, noun, adjective, verb any of which may be None if there was 
    no good match
    """
    pronoun = None
    noun = None
    adjective = None
    verb = None
    for sent in parsed.sentences:
        pronoun = find_pronoun(sent)
        noun = find_noun(sent)
        adjective = find_adjective(sent)
        verb = find_verb(sent)
    logger.info("Pronoun=%s, noun=%s, adjective=%s, verb=%s", pronoun, noun, adjective, verb)
    return pronoun, noun, adjective, verb

def find_pronoun(sent):
    """
    Given a sentence (TextBlob Sentence object), find a preferred pronoun to respond with. 
    Returns None if no candidate pronoun is found in the input
    """
    pronoun = None

    for word, part_of_speech in sent.pos_tags:
        # Disambiguate pronouns
        if part_of_speech == 'PRP' and word.lower() == 'you':
            pronoun = 'I'
        elif part_of_speech == 'PRP' and word.lower() == 'i':
            # If the user mentioned themselves, then they will definitely be the pronoun
            pronoun = 'You'
    return pronoun

def find_noun(sent):
    """
    Given a sentence (TextBlob Sentence object), find the best candidate noun.
    """
    noun = None

    if not noun:
        for w, p in sent.pos_tags:
            if p == 'NN':  # This is a noun
                noun = w
                break
    if noun:
        logger.info("Found noun: %s", noun)

    return noun

def find_adjective(sent):
    """
    Given a sentence (TextBlob Sentence object), find the best candidate adjective.
    """
    adj = None
    for w, p in sent.pos_tags:
        if p == 'JJ':  # This is an adjective
            adj = w
            break
    return adj

def find_verb(sent):
    """
    Pick a candidate verb for the sentence.
    """
    verb = None
    pos = None
    for word, part_of_speech in sent.pos_tags:
        if part_of_speech.startswith('VB'):  # This is a verb
            verb = word
            pos = part_of_speech
            break
    return verb, pos



In [ ]:
# Examples
sentences = processed_text.sentences
print('sentences[0]:',sentences[0], '\nsentences[1]:', sentences[1])

print("\nResponse Pronouns")
print(find_pronoun(sentences[0]), find_pronoun(sentences[1]))

print("\nNouns")
print(find_noun(sentences[0]), find_noun(sentences[1]))

print("\nAdjectives")
print(find_adjective(sentences[0]), find_adjective(sentences[1]))

print("\nVerbs")
print(find_verb(sentences[0]), find_verb(sentences[1]))

print("\nAltogether Now!")
pronoun, noun, adjective, verb = find_candidate_parts_of_speech(processed_text)
print("Pronoun:", pronoun, "\nNoun:", noun, "\nAdjective:", adjective, "\nVerb", verb)

# Differential Response
The chatbot does not have to be able to respond to every type of input possible.
However, it's desirable to differentiate a greeting from a question, and so on.
In this regard, we can design several functions to determine some additional context
of the input text and decide what type of response is likely to make the most sense.

### Respond to a Greeting
The simplest way to approach a greeting is rules-based: make a list of likely greetings and a list of potential responses. Since this is a stateless chatbot, it won't get annoyed if you say "hi" 10,000 times! It picks its response at random. In practice, the tutorial advises to cycle through the responses, or at the least hedge against immediate repeats.

In [ ]:
# Sentences we'll respond with if the user greeted us.
GREETING_KEYWORDS = ("hello", "hi", "greetings", "sup", "what's up", "hola")

GREETING_RESPONSES = [
    "What do ya want?",
    "Let's make this quick. I only got about two humps left in me.",
    "You must want something.",
    "Hey, I was just about to go get some coffee.",
    "Huh? Get me that wrench.",
    "Wut da hecckkk?",
    "I need a coffee.",
]

def check_for_greeting(sentence):
    """If any of the words in the user's input was a greeting, return a greeting response."""
    for word in sentence.words:
        if word.lower() in GREETING_KEYWORDS:
            return random.choice(GREETING_RESPONSES)


In [ ]:
print(check_for_greeting(sentences[0]))
print(check_for_greeting(sentences[1]))

### Repond to Comments about BillyBot

In [ ]:
SELF_VERBS_WITH_NOUN_CAPS_PLURAL = [
    "{noun}? I've already mowed the grass and washed my car.",
    "{noun} are like used cars: a damn headache.",
    "I seen enough {noun} to know it ain't fishing.",
    "{noun} don't bother me, as long as I can smoke my cigarettes and drink my coffee.",
]

SELF_VERBS_WITH_NOUN_LOWER = [
    "Yeah, but I can probably fix a {noun} with a wrench and some duct tape.",
    "I know enough about {noun} to know I rather be fishing.",
    "Ask me about a {noun} after I finish this coffee.",
    "That {noun} reminds me of a prick customer I had when I was fixin' cars.",
    "This must be the place.",
    "Is time for a coffee yet?",
]

SELF_VERBS_WITH_ADJECTIVE = [
    "Not as {adjective} as the last bass I caught.",
    "I like my coffee like I like my fish: {adjective}.",
    "{adjective}? Wut da hecckkk?",
    "If your head wasn't screwed on, I swear ya'd lose the thing.",
    "It don't have to be {adjective} perfect.",
]

def check_for_comment_about_bot(pronoun, noun, adjective):
    """Check if the user's input was about the bot itself, in which case try to fashion a response
    that feels right based on their input. Returns the new best sentence, or None."""
    resp = None
    if pronoun is not None:
        if pronoun.lower() == 'i' and (noun or adjective):
            if noun:
                if random.choice((True, False)):
                    resp = random.choice(SELF_VERBS_WITH_NOUN_CAPS_PLURAL).format(**{'noun': noun.pluralize().capitalize()})
                else:
                    resp = random.choice(SELF_VERBS_WITH_NOUN_LOWER).format(**{'noun': noun})
            else:
                resp = random.choice(SELF_VERBS_WITH_ADJECTIVE).format(**{'adjective': adjective})
    return resp

In [ ]:
example_text = process_input_text('You are so cool.')
pronoun, noun, adjective, verb = find_candidate_parts_of_speech(example_text)
print(check_for_comment_about_bot(pronoun, noun, adjective))

### Respond to Any Other Type of Input Text

In [ ]:
def construct_response(pronoun, noun, verb):
    """
    No special cases matched, so we're going to try to construct a full sentence that uses as much
    of the user's input as possible.
    """
    resp = []

    if pronoun:
        resp.append(pronoun)

    # We always respond in the present tense, and the pronoun will always either be a passthrough
    # from the user, or 'you' or 'I', in which case we might need to change the tense for some
    # irregular verbs.
    if verb:
        verb_word = verb[0]
        if verb_word in ('be', 'am', 'is', "'m"):  # This would be an excellent place to use lemmas!
            if pronoun.lower() == 'you':
                # The bot will always tell the person they aren't whatever they said they were.
                resp.append("aren't really")
            else:
                resp.append(verb_word)
    if noun:
        article = "an" if starts_with_vowel(noun) else "a"
        resp.append(article + " " + noun)

    resp.append(random.choice(("I reckon", "near as I can tell", "if you say so", "before supper", "")))

    return " ".join(resp)


# Sentences we'll respond with if we have no idea what the user just said.
NONE_RESPONSES = [
    "This must be the place.",
    "Wut da hecckkk?",
    "Go grab me that 9/16 wrench and ask me again.",
    "I was just about to get some coffee.",
    "You coming? No, just breathing heavy.",
    "I caught a bass once that made more sense than that.",
]


### Explicit/Toxic Content Filtering

In [ ]:
def filter_response(resp):
    """Don't allow any words to match our filter list"""
    tokenized = resp.split(' ')
    for word in tokenized:
        if '@' in word or '#' in word or '!' in word:
            raise UnacceptableUtteranceException()
        for s in FILTER_WORDS:
            if word.lower().startswith(s):
                raise UnacceptableUtteranceException()

### Don't Sound Stupid
...unless you want BillyBot to, in which case you can make a function that encourages BillyBot to sound stupid.

Here, we check for words that start with vowels to help avoid BillyBot saying things like "a apple" instead of "an apple." 

In [ ]:
def starts_with_vowel(word):
    """Check for pronoun compability -- 'a' vs. 'an'"""
    return True if word[0] in 'aeiou' else False

## Response Function

In [ ]:
def respond(input_text):
    """
    Parse the user's inbound sentence and find candidate terms that make up a best-fit response
    """
    cleaned = preprocess_text(input_text)
    parsed = TextBlob(cleaned)

    # Loop through all the sentences, if more than one. This will help extract the most relevant
    # response text even across multiple sentences (for example if there was no obvious direct noun
    # in one sentence
    pronoun, noun, adjective, verb = find_candidate_parts_of_speech(parsed)

    # If we said something about the bot and used some kind of direct noun, construct the
    # sentence around that, discarding the other candidates
    resp = check_for_comment_about_bot(pronoun, noun, adjective)

    # If we just greeted the bot, we'll use a return greeting
    if not resp:
        resp = check_for_greeting(parsed)

    if not resp:
        # If we didn't override the final sentence, try to construct a new one:
        if not pronoun:
            resp = random.choice(NONE_RESPONSES)
        elif pronoun == 'I' and not verb:
            resp = random.choice(COMMENTS_ABOUT_SELF)
        else:
            resp = construct_response(pronoun, noun, verb)

    # If we got through all that with nothing, use a random response
    if not resp:
        resp = random.choice(NONE_RESPONSES)

    logger.info("Returning phrase '%s'", resp)
    # Check that we're not going to say anything obviously offensive
    filter_response(resp)

    return resp
